In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 1. Load dataset
df = pd.read_csv("uber.csv")

# 2. Preprocessing: remove unnecessary columns and missing/invalid records
df.drop(["Unnamed: 0", "key"], axis=1, inplace=True, errors="ignore")
df.dropna(inplace=True)
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])

df = df[
    df["pickup_latitude"].between(40, 42) &
    df["dropoff_latitude"].between(40, 42) &
    df["pickup_longitude"].between(-75, -72) &
    df["dropoff_longitude"].between(-75, -72) &
    (df["fare_amount"] > 0) &
    df["passenger_count"].between(1, 6)
].copy()

# Extract date/time features
df["hour"] = df["pickup_datetime"].dt.hour
df["day"] = df["pickup_datetime"].dt.day
df["month"] = df["pickup_datetime"].dt.month
df["year"] = df["pickup_datetime"].dt.year
df["dayofweek"] = df["pickup_datetime"].dt.dayofweek

# 3. Approximate pickup-to-dropoff distance using a local Euclidean formula
lat_diff = df["dropoff_latitude"] - df["pickup_latitude"]
lon_diff = df["dropoff_longitude"] - df["pickup_longitude"]
mean_lat = np.radians((df["pickup_latitude"] + df["dropoff_latitude"]) / 2)
df["distance_km"] = 111.32 * np.sqrt(lat_diff**2 + (lon_diff * np.cos(mean_lat))**2)

# 4. Outlier detection and IQR capping
# Boxplots display the distributions before capping.
df[["fare_amount", "distance_km"]].boxplot()
plt.show()

for col in ["fare_amount", "distance_km"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    df[col] = df[col].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

# 5. Correlation analysis
sns.heatmap(df.select_dtypes("number").corr())
plt.show()

# 6. Features and target
X = df[["pickup_longitude", "pickup_latitude", "dropoff_longitude",
        "dropoff_latitude", "passenger_count", "hour", "day", "month",
        "year", "dayofweek", "distance_km"]]
y = df["fare_amount"]

# 7. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 8. Train both regression models
lr = LinearRegression()
rf = RandomForestRegressor(n_estimators=10, random_state=42)
lr.fit(X_train, y_train)
rf.fit(X_train, y_train)

# 9. Evaluate both models
for name, model in [("Linear Regression", lr), ("Random Forest", rf)]:
    pred = model.predict(X_test)
    print("\n", name)
    print("R2:", r2_score(y_test, pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
    print("MAE:", mean_absolute_error(y_test, pred))
